# RoadSegBench — IJIES Reproducibility Pipeline (Google Colab)

**Purpose:** Rebuild the benchmark from frozen manifests and author-supplied datasets/checkpoints. This notebook does not contain proprietary datasets or precomputed results.

**Review-driven safeguards:** exact population accounting, disjoint partitions, curvature extractor configuration, metric consistency, paired statistical inputs, and auditable output files.

Run cells in order. Do not skip the manifest audit.

## 1. Runtime setup

In [ ]:
# Optional: mount Google Drive to access licensed datasets and experiment artifacts.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/YOUR-USERNAME/RoadSegBench-IJIES.git /content/RoadSegBench-IJIES
%cd /content/RoadSegBench-IJIES
!pip install -r requirements-colab.txt
!pip install -e .

## 2. Record environment and code revision

In [ ]:
import os, sys, json, subprocess, platform
from pathlib import Path
import numpy, pandas, scipy, cv2, torch

def git_revision():
    try:
        return subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
    except Exception:
        return 'UNAVAILABLE: run from a git checkout'

environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': numpy.__version__,
    'pandas': pandas.__version__,
    'scipy': scipy.__version__,
    'opencv': cv2.__version__,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'git_commit': git_revision()
}
Path('outputs').mkdir(exist_ok=True)
Path('outputs/environment.json').write_text(json.dumps(environment, indent=2))
environment

## 3. Configure dataset roots and load the frozen manifest

In [ ]:
import pandas as pd
from pathlib import Path
from roadsegbench.manifests import load_manifest, validate_paths, overlap_report

MANIFEST = 'data/manifests/roadsegbench_manifest.csv'  # author must create this
DATASET_ROOTS = {
    'cityscapes': '/content/drive/MyDrive/datasets/cityscapes',
    'bdd100k': '/content/drive/MyDrive/datasets/bdd100k',
    'mapillary': '/content/drive/MyDrive/datasets/mapillary'
}
assert Path(MANIFEST).exists(), (
    'Create the frozen manifest first. See data/manifests/README.md and README.md.'
)
manifest = load_manifest(MANIFEST)
print('Total manifest rows:', len(manifest))
display(manifest.groupby(['dataset','split','partition']).size().rename('count').reset_index())

In [ ]:
path_errors = validate_paths(manifest, DATASET_ROOTS)
print('Missing path count:', len(path_errors))
for e in path_errors[:30]:
    print(e)
if path_errors:
    raise RuntimeError('Resolve missing image/mask paths before proceeding.')

## 4. Partition leakage and population reconciliation

In [ ]:
overlaps = overlap_report(manifest)
display(overlaps)
if (overlaps.overlap_count > 0).any():
    raise RuntimeError('Partition overlap detected. Fix image-level leakage before training/evaluation.')

In [ ]:
# Explicitly test the two reviewer-mentioned Cityscapes Sharp populations.
# Supply exact image_id lists/filters from the original Table 5a and urban-control analysis.
# Do not type the expected 97 or 214 as if they were verified: derive them from frozen IDs.
def compare_named_populations(manifest, ids_a, ids_b, name_a='main_table_sharp',
                              name_b='urban_control_sharp'):
    a, b = set(ids_a), set(ids_b)
    return {
        'population_a': name_a, 'population_b': name_b,
        'n_a': len(a), 'n_b': len(b), 'intersection': len(a & b),
        'b_is_subset_of_a': b.issubset(a),
        'b_not_in_a': sorted(b-a)[:20]
    }

# Example after author supplies lists:
# print(compare_named_populations(manifest, main_sharp_ids, urban_sharp_ids))

## 5. Curvature extractor smoke test and tier assignment

In [ ]:
from roadsegbench.curvature import controlled_curve_smoke_test, mask_boundary_curvature, classify_curvature
import matplotlib.pyplot as plt

synthetic_scores = controlled_curve_smoke_test()
print('Synthetic shape curvature smoke test:', synthetic_scores)

In [ ]:
# Compute curvature for all masks in the frozen manifest.
# IMPORTANT: confirm the mask decoding/road-class mapping for each dataset before running.
import cv2
from tqdm.auto import tqdm

GENTLE_UPPER = 0.010
MODERATE_UPPER = 0.030

scores, tiers, errors = [], [], []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest)):
    mask_path = Path(DATASET_ROOTS[row.dataset]) / row.mask_path
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if mask is None:
        scores.append(float('nan')); tiers.append('ERROR')
        errors.append((row.image_id, 'mask_read_failed'))
        continue
    # TODO REQUIRED: replace this generic nonzero mapping with the verified
    # road-class ID mapping for the specific dataset and label format.
    # For Cityscapes train IDs, road is commonly trainId 0; verify your files.
    road_binary = (mask == 0).astype('uint8') if mask.ndim == 2 else (mask[:,:,0] > 0).astype('uint8')
    try:
        score = mask_boundary_curvature(road_binary)
        scores.append(score)
        tiers.append(classify_curvature(score, GENTLE_UPPER, MODERATE_UPPER))
    except Exception as exc:
        scores.append(float('nan')); tiers.append('ERROR')
        errors.append((row.image_id, str(exc)))

manifest['curvature'] = scores
manifest['curvature_tier'] = tiers
manifest.to_csv('data/manifests/roadsegbench_manifest_with_curvature.csv', index=False)
pd.DataFrame(errors, columns=['image_id','error']).to_csv('outputs/curvature_errors.csv', index=False)
display(manifest.groupby(['dataset','split','partition','curvature_tier']).size().rename('count').reset_index())

## 6. Tier percentages and boundary-weighting audit

In [ ]:
tier_counts = (manifest[manifest.partition == 'evaluation']
               .groupby('curvature_tier').size().rename('count').reset_index())
tier_counts['empirical_percent'] = 100 * tier_counts['count'] / tier_counts['count'].sum()
display(tier_counts)

# Choose exactly one documented weighting rule for the manuscript.
# Empirical composition uses observed evaluation counts. Standardized weights
# must be explicitly labelled as standardized and must not be described as empirical.
WEIGHTING_RULE = 'empirical'  # or 'standardized'
STANDARDIZED_WEIGHTS = {'Gentle': 0.50, 'Moderate': 0.33, 'Sharp': 0.17}

if WEIGHTING_RULE == 'empirical':
    weights = dict(zip(tier_counts.curvature_tier,
                       tier_counts['count'] / tier_counts['count'].sum()))
elif WEIGHTING_RULE == 'standardized':
    weights = STANDARDIZED_WEIGHTS
else:
    raise ValueError('Unknown weighting rule')
print('Documented tier weights:', weights)

## 7. Reference U-Net training (executable baseline)

This trains a transparent binary-road U-Net baseline. It does **not** reproduce DeepLabV3+, SegFormer, PIDNet, DDRNet, BiSeNetV2, or SCTNet; integrate exact architecture variants and settings separately before making comparative claims.


In [ ]:
from roadsegbench.training import train_unet
from pathlib import Path

# REQUIRED: implement a verified decoder for each dataset's label IDs.
# This example is only appropriate if road pixels are encoded as class ID 0.
def verified_road_decoder(mask_array, dataset):
    if mask_array.ndim != 2:
        raise ValueError(f'Expected indexed 2-D label mask for {dataset}; implement its decoder.')
    return (mask_array == 0).astype('uint8')

train_df = manifest[manifest.partition == 'train'].copy()
tuning_df = manifest[manifest.partition == 'tuning'].copy()
assert len(train_df) and len(tuning_df), 'Train and tuning partitions must be non-empty.'
model, history, best_iou = train_unet(
    train_df, tuning_df, DATASET_ROOTS, 'outputs/checkpoints/unet_seed11',
    road_mask_decoder=verified_road_decoder, epochs=50, batch_size=4,
    lr=1e-4, size=(512,1024), seed=11, workers=2)
print('Best tuning road IoU:', best_iou)


## 7. Segmentation metrics from saved predictions

In [ ]:
from roadsegbench.metrics import binary_metrics, boundary_iou
# Expected prediction manifest: image_id, model, seed, prediction_path, target_path.
PREDICTIONS_CSV = 'outputs/predictions_manifest.csv'
if Path(PREDICTIONS_CSV).exists():
    pred_df = pd.read_csv(PREDICTIONS_CSV)
    metric_rows = []
    for r in tqdm(pred_df.itertuples(index=False), total=len(pred_df)):
        pred = cv2.imread(r.prediction_path, cv2.IMREAD_GRAYSCALE)
        target = cv2.imread(r.target_path, cv2.IMREAD_GRAYSCALE)
        if pred is None or target is None:
            raise FileNotFoundError(f'Cannot read prediction/target for {r.image_id}')
        pred01 = pred.astype('float32') / 255.0
        target01 = (target > 0).astype('uint8')
        m = binary_metrics(pred01, target01, threshold=0.5)
        m['biou'] = boundary_iou(pred01, target01, threshold=0.5, radius=2)
        metric_rows.append({**r._asdict(), **m})
    image_metrics = pd.DataFrame(metric_rows)
    image_metrics.to_csv('outputs/per_image_metrics.csv', index=False)
    display(image_metrics.head())
else:
    print('Add outputs/predictions_manifest.csv after model inference. No metrics are fabricated.')

## 8. Statistical reanalysis from paired run-level inputs

In [ ]:
from roadsegbench.statistics import paired_normal_comparisons, nemenyi_cd_decisions, friedman_test
SEED_METRICS = 'outputs/seed_level_metrics.csv'
if Path(SEED_METRICS).exists():
    seed_df = pd.read_csv(SEED_METRICS)
    MODEL_COLUMNS = [c for c in seed_df.columns if c not in {'seed','scene_id','dataset','tier'}]
    # Ensure rows are genuinely paired on the same seed and evaluation population.
    if seed_df.duplicated(subset=['seed']).any():
        print('Warning: multiple rows per seed. Define paired unit explicitly (e.g., seed+scene_id).')
    comparisons = paired_normal_comparisons(seed_df, MODEL_COLUMNS)
    comparisons.to_csv('outputs/paired_holm_comparisons.csv', index=False)
    display(comparisons)
else:
    print('Provide seed_level_metrics.csv with paired seed-level endpoint values.')

In [ ]:
# Nemenyi: enter recomputed mean ranks from the complete Friedman/Nemenyi analysis.
# This function corrects pairwise interpretation against the supplied CD.
# Do not hardcode a CD unless it is justified by the chosen test and number of models/blocks.
MEAN_RANKS = None  # e.g. {'U-Net': ..., 'DeepLabV3+': ...}
CRITICAL_DIFFERENCE = None
if MEAN_RANKS is not None and CRITICAL_DIFFERENCE is not None:
    decisions = nemenyi_cd_decisions(MEAN_RANKS, CRITICAL_DIFFERENCE)
    decisions.to_csv('outputs/nemenyi_pairwise_decisions.csv', index=False)
    display(decisions)

## 9. Threshold sensitivity and matched-scene analysis

In [ ]:
# Expected threshold-sweep CSV columns:
# image_id, model, threshold, tier, scene_id, road_iou, biou
THRESHOLD_CSV = 'outputs/threshold_sweep_per_image.csv'
if Path(THRESHOLD_CSV).exists():
    th = pd.read_csv(THRESHOLD_CSV)
    summary = (th.groupby(['model','threshold','tier'])
               .agg(n_images=('image_id','nunique'),
                    mean_road_iou=('road_iou','mean'),
                    mean_biou=('biou','mean')).reset_index())
    summary.to_csv('outputs/threshold_sweep_summary.csv', index=False)
    display(summary)
else:
    print('Run inference at every prespecified threshold and provide per-image outputs.')

## 10. Latency protocol

In [ ]:
import time, numpy as np, torch

def benchmark_latency(model, sample, warmup=50, iterations=200, device=None):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device).eval()
    sample = sample.to(device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(sample)
        if device.startswith('cuda'):
            torch.cuda.synchronize()
        times = []
        for _ in range(iterations):
            t0 = time.perf_counter()
            _ = model(sample)
            if device.startswith('cuda'):
                torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)*1000)
    return {'device': device, 'warmup': warmup, 'iterations': iterations,
            'mean_ms': float(np.mean(times)), 'median_ms': float(np.median(times)),
            'p95_ms': float(np.percentile(times,95)),
            'fps_batch1': float(1000/np.mean(times))}
print('Use identical hardware, input size, precision, backend and timing protocol for all compared models.')

## 11. Reproducibility checklist and final artifact export

In [ ]:
checklist = {
    'frozen_manifest_committed': Path('data/manifests/roadsegbench_manifest.csv').exists(),
    'curvature_manifest_generated': Path('data/manifests/roadsegbench_manifest_with_curvature.csv').exists(),
    'environment_recorded': Path('outputs/environment.json').exists(),
    'per_image_metrics_available': Path('outputs/per_image_metrics.csv').exists(),
    'seed_level_metrics_available': Path('outputs/seed_level_metrics.csv').exists(),
    'predictions_manifest_available': Path('outputs/predictions_manifest.csv').exists(),
}
display(pd.DataFrame([{'requirement':k,'present':v} for k,v in checklist.items()]))
print('Commit only verified outputs. Do not commit restricted datasets or credentials.')

## 12. Required author actions before public release

- Replace the generic mask decoding cell with verified class-ID mappings for each dataset.
- Validate curvature against controlled curves, resampling perturbations, centerline/direction measures, holes, intersections, border truncation, and degenerate masks.
- Supply exact source-specific manifests and reconcile the reviewer-identified 97-versus-214 Sharp subsets.
- Add exact model implementations/configurations and training code/checkpoint metadata for every reported architecture.
- Generate per-image and per-seed records from actual inference runs.
- Recompute all tables/figures from these records and ensure manuscript values match repository outputs.
- Include `Conflicts of Interest` and `Author Contributions` in the manuscript as requested by the editor.

## 13. Multi-dataset / multi-seed training matrix

The new package supports source-specific training runs (Cityscapes, BDD100K, Mapillary) and a model registry. Train each dataset separately unless the manuscript explicitly defines a joint-training protocol. This prevents an ambiguous pooled checkpoint definition.

Before running, edit each dataset YAML with verified mask encoding and class IDs. Do not use the placeholder BDD100K or Mapillary class IDs.

In [ ]:
# Example: Cityscapes U-Net, ten independent seeds.
!python scripts/train_matrix.py \\
  --manifest data/manifests/roadsegbench_manifest_with_curvature.csv \\
  --dataset-config configs/datasets/cityscapes.yaml \\
  --architecture unet \\
  --model-config '{"encoder":"resnet34","encoder_weights":"imagenet"}' \\
  --seeds 11,22,33,44,55,66,77,88,99,111 \\
  --epochs 60 --out outputs/runs

## 14. Architecture coverage and exact-variant policy

Native registry implementations currently cover U-Net, DeepLabV3+, SegFormer, FPN and PSPNet. The registry deliberately refuses to label a substitute network as PIDNet, DDRNet, BiSeNetV2 or SCTNet. To complete those four manuscript-specific models, add adapters to the exact upstream source revision and record repository URL, commit SHA, license, config file, pretrained checkpoint checksum, teacher/distillation settings (especially SCTNet), and preprocessing. Then implement a factory branch in `roadsegbench/models/registry.py` and run parity tests.

This is intentional: the reviewer explicitly questioned the SCTNet reference and training-only teacher/loss details. A different network under the same model name would make the repository scientifically misleading.

In [ ]:
# Verify model coverage before starting a full benchmark.
from roadsegbench.models.registry import AVAILABLE_NATIVE, REQUIRES_PINNED_ADAPTER
print("Available native implementations:", AVAILABLE_NATIVE)
print("Requires exact upstream adapter:", REQUIRES_PINNED_ADAPTER)